<a href="https://colab.research.google.com/github/Pawan-model/Huggingface-Experiment/blob/main/Manual_Training_loops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install transformers[sentencepiece]
!pip install evaluate

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding,AutoModelForSequenceClassification
checkpoint='bert-base-uncased'
raw_dataset=load_dataset('glue','mrpc')
tokenizer=AutoTokenizer.from_pretrained(checkpoint)
model=AutoModelForSequenceClassification.from_pretrained(checkpoint)

In [ ]:
def tokenize_function(example):
  return tokenizer(example['sentence1'],example['sentence2'],truncation=True)
tokenized_dataset=raw_dataset.map(tokenize_function,batched=True)
data_collator=DataCollatorWithPadding(tokenizer=tokenizer)

Prepare for training

In [ ]:
## Preparing the datasets by first removing unwanted columns , renaming columns label to labels(as model expects this argumnet) and seting the format to torch so that it returns tensor insted of lists
tokenized_dataset = tokenized_dataset.remove_columns(['sentence1','sentence2','idx'])
tokenized_dataset= tokenized_dataset.rename_column('label','labels')
tokenized_dataset.set_format('torch')
tokenized_dataset['train'].column_names

Training loop

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_scheduler
from accelerate import Accelerator
accelerator=Accelerator()
train_dataloader= DataLoader(tokenized_dataset['train'],shuffle=True,batch_size=8,collate_fn=data_collator)
eval_dataloader=DataLoader(tokenized_dataset['validation'],batch_size=8,collate_fn=data_collator)
optimizer=AdamW(model.parameters(), lr=5e-5)
model,optimizer,train_dataloader,eval_dataloader=accelerator.prepare(model,optimizer,train_dataloader,eval_dataloader)
num_epochs=3
num_steps=num_epochs*len(train_dataloader)
lr_scheduler=get_scheduler('linear',
                           optimizer=optimizer,
                           num_training_steps=num_steps,
                           num_warmup_steps=0,
                           )
print(num_steps)

In [ ]:
from tqdm.auto import tqdm
progress_bar=tqdm(range(num_steps))
model.train()
for epoch in range(num_epochs):
  for batch in train_dataloader:
    output=model(**batch)
    loss=output.loss
    accelerator.backward(loss)
    optimizer.step()
    lr_scheduler.step()
    optimizer.zero_grad()
    progress_bar.update(1)


Evaluation loop

In [ ]:
import evaluate
metric=evaluate.load('glue','mrpc')
model.eval()
for batch in eval_dataloader:
  with torch.no_grad():
    outputs=model(**batch)
  logits=outputs.logits
  predictions=torch.argmax(logits,dim=-1)
  metric.add_batch(predictions=predictions,references=batch['labels'])
metric.compute()
